# Sistema de Recomendação Multimodal — v3

Combina **três modalidades** para gerar as recomendações:

| Modalidade | Arquivo | Dimensões | Captura |
|---|---|---|---|
| **Texto (TF-IDF)** | `animes_tfidf.csv` | 2000 | Temática, enredo, personagens |
| **Imagem (RGB hist.)** | `banco_de_dados_animes_vetorizado.pkl` | 26 | Paleta de cores, brilho |
| **Gênero (one-hot)** | coluna `Genres` do pkl | 19 | Gêneros MAL |

Similaridade final:
```
sim = peso_texto × sim_tfidf + peso_imagem × sim_imagem + peso_genero × sim_genero
```

Além do cosseno multimodal, o notebook oferece mais 3 métricas: **Euclidiana**, **Manhattan** e **Jaccard**.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances, manhattan_distances
from sklearn.preprocessing import normalize

In [2]:
# ── 1. Carregamento ───────────────────────────────────────────────
print("Carregando dados...")

df_base  = pd.read_pickle('banco_de_dados_animes_vetorizado.pkl')   # metadados + vetor_imagem
df_tfidf = pd.read_csv('animes_tfidf.csv')                          # metadados + colunas tfidf_*

# Garante mesma ordem de linhas nas duas fontes
df_tfidf = df_tfidf.set_index('Title').loc[df_base['Title'].values].reset_index()

assert list(df_base['Title']) == list(df_tfidf['Title']), "Títulos desalinhados!"
print(f"OK — {len(df_base)} animes carregados.")

Carregando dados...
OK — 1050 animes carregados.


/var/folders/b0/d0xbbxmd447fjfd0jsx10zrm0000gn/T/ipykernel_72947/3726352362.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tfidf = df_tfidf.set_index('Title').loc[df_base['Title'].values].reset_index()


In [3]:
# ── 2. Matrizes de vetores (normalizadas L2) ──────────────────────
print("Preparando matrizes...")

# Texto
colunas_tfidf  = [c for c in df_tfidf.columns if c.startswith('tfidf_')]
matriz_tfidf   = normalize(df_tfidf[colunas_tfidf].values)

# Imagem
matriz_imagem  = normalize(np.stack(df_base['vetor_imagem'].values))

# Gênero  (one-hot a partir da coluna 'Genres')
genres_dummies = df_base['Genres'].fillna('').str.get_dummies(sep=', ')
matriz_generos = normalize(genres_dummies.values.astype(float))
all_genres     = list(genres_dummies.columns)

print(f"  TF-IDF : {matriz_tfidf.shape}")
print(f"  Imagem : {matriz_imagem.shape}")
print(f"  Gêneros: {matriz_generos.shape}  →  {all_genres}")

Preparando matrizes...
  TF-IDF : (1050, 2000)
  Imagem : (1050, 26)
  Gêneros: (1050, 19)  →  ['Action', 'Adventure', 'Avant Garde', 'Award Winning', 'Boys Love', 'Comedy', 'Drama', 'Ecchi', 'Fantasy', 'Girls Love', 'Gourmet', 'Horror', 'Mystery', 'Romance', 'Sci-Fi', 'Slice of Life', 'Sports', 'Supernatural', 'Suspense']


In [4]:
# ── 3. Pré-cálculo das matrizes de similaridade ───────────────────
# (calculadas uma vez; combinação feita por pesos em tempo de consulta)

print("Calculando similaridades...")

# Cosseno por modalidade
sim_cos_texto   = cosine_similarity(matriz_tfidf)
sim_cos_imagem  = cosine_similarity(matriz_imagem)
sim_cos_generos = cosine_similarity(matriz_generos)

# Euclidiana (convertida em similaridade: quanto menor a dist, maior o score)
sim_eucl_texto  = 1 / (1 + euclidean_distances(matriz_tfidf))
sim_eucl_imagem = 1 / (1 + euclidean_distances(matriz_imagem))
sim_eucl_generos= 1 / (1 + euclidean_distances(matriz_generos))

# Manhattan
sim_manh_texto  = 1 / (1 + manhattan_distances(matriz_tfidf))
sim_manh_imagem = 1 / (1 + manhattan_distances(matriz_imagem))
sim_manh_generos= 1 / (1 + manhattan_distances(matriz_generos))

# Jaccard (versão binarizada: presença / ausência do termo)
def jaccard_matrix(X):
    X_bin    = (X > 0).astype(float)
    inter    = X_bin @ X_bin.T
    row_sums = X_bin.sum(axis=1)
    uniao    = row_sums[:, None] + row_sums[None, :] - inter
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(uniao == 0, 0.0, inter / uniao)

sim_jacc_texto   = jaccard_matrix(matriz_tfidf)
sim_jacc_imagem  = jaccard_matrix(matriz_imagem)
sim_jacc_generos = jaccard_matrix(matriz_generos)

# Agrupa as matrizes por métrica para facilitar a combinação
MATRIZES = {
    'Cosseno':    (sim_cos_texto,   sim_cos_imagem,   sim_cos_generos),
    'Euclidiana': (sim_eucl_texto,  sim_eucl_imagem,  sim_eucl_generos),
    'Manhattan':  (sim_manh_texto,  sim_manh_imagem,  sim_manh_generos),
    'Jaccard':    (sim_jacc_texto,  sim_jacc_imagem,  sim_jacc_generos),
}

print("Pronto!")

Calculando similaridades...
Pronto!


In [12]:
# ── 4. Função de Recomendação ─────────────────────────────────────

def recomendar(
    titulo,
    df,
    matrizes,
    peso_texto   = 0.80,
    peso_imagem  = 0.10,
    peso_genero  = 0.10,
    metrica      = None,   # None = todas | 'Cosseno' | 'Euclidiana' | 'Manhattan' | 'Jaccard'
    n            = 10,
    filtro_genero= None,   # lista de gêneros para restringir candidatos, ex: ['Action', 'Fantasy']
):
    """
    Recomenda os N animes mais similares usando combinação multimodal.

    Parâmetros
    ----------
    titulo        : str   — título exato (coluna 'Title')
    df            : DataFrame com colunas 'Title' e 'Genres'
    matrizes      : dict  — {'Metrica': (sim_texto, sim_imagem, sim_genero)}
    peso_texto    : float — peso da similaridade textual  (devem somar 1.0)
    peso_imagem   : float — peso da similaridade visual
    peso_genero   : float — peso da similaridade de gênero
    metrica       : str | None — filtra para exibir só uma métrica
    n             : int   — número de recomendações
    filtro_genero : list | None — restringe candidatos a animes com ao menos um gênero da lista

    Retorna
    -------
    dict[metrica] → DataFrame com colunas Title, Genres, sim_texto, sim_imagem, sim_genero, sim_final
    """
    assert abs(peso_texto + peso_imagem + peso_genero - 1.0) < 1e-6, \
        f"Pesos devem somar 1.0. Atual: {peso_texto+peso_imagem+peso_genero:.4f}"

    if titulo not in df['Title'].values:
        print(f"⚠️  '{titulo}' não encontrado no banco de dados.")
        return {}

    idx = df[df['Title'] == titulo].index[0]

    # Índices candidatos (exclui o próprio anime)
    candidatos = [i for i in range(len(df)) if i != idx]

    # Filtro por gênero
    if filtro_genero:
        filtro_set = set(filtro_genero)
        candidatos = [
            i for i in candidatos
            if isinstance(df['Genres'].iloc[i], str) and (filtro_set & set(df['Genres'].iloc[i].split(', ')))
        ]
        if not candidatos:
            print(f"⚠️  Nenhum anime encontrado com os gêneros: {filtro_genero}")
            return {}

    metricas_usar = {metrica: matrizes[metrica]} if metrica else matrizes
    resultados = {}

    for nome, (sT, sI, sG) in metricas_usar.items():
        # Score combinado
        scores_comb = (
            peso_texto  * sT[idx] +
            peso_imagem * sI[idx] +
            peso_genero * sG[idx]
        )

        top = sorted(candidatos, key=lambda i: scores_comb[i], reverse=True)[:n]

        df_res = pd.DataFrame([
            {
                'Title':      df['Title'].iloc[i],
                'Genres':     df['Genres'].iloc[i],
                'sim_texto':  round(float(sT[idx][i]),  4),
                'sim_imagem': round(float(sI[idx][i]),  4),
                'sim_genero': round(float(sG[idx][i]),  4),
                'sim_final':  round(float(scores_comb[i]), 4),
            }
            for i in top
        ])

        resultados[nome] = df_res

        # Exibe
        print(f"\n{'='*68}")
        print(f"  [{nome}]  Top {n} para '{titulo}'")
        print(f"  Pesos → Texto: {peso_texto:.0%} | Imagem: {peso_imagem:.0%} | Gênero: {peso_genero:.0%}")
        if filtro_genero:
            print(f"  Filtro gênero: {filtro_genero}  ({len(candidatos)} candidatos)")
        print(f"{'='*68}")
        print(f"  {'#':>2}  {'Título':<42}  {'Txt':>5}  {'Img':>5}  {'Gen':>5}  {'Final':>6}")
        print(f"  {'─'*66}")
        for rank, row in enumerate(df_res.itertuples(), 1):
            print(f"  {rank:>2}. {row.Title:<42}  "
                  f"{row.sim_texto:.3f}  {row.sim_imagem:.3f}  "
                  f"{row.sim_genero:.3f}  {row.sim_final:.4f}")

    return resultados

In [13]:
# ── 5. Demonstração — todas as métricas ──────────────────────────
resultados = recomendar(
    'Sousou no Frieren',
    df_base,
    MATRIZES,
    peso_texto  = 0.80,
    peso_imagem = 0.10,
    peso_genero = 0.10,
    n = 10,
)


  [Cosseno]  Top 10 para 'Sousou no Frieren'
  Pesos → Texto: 80% | Imagem: 10% | Gênero: 10%
   #  Título                                        Txt    Img    Gen   Final
  ──────────────────────────────────────────────────────────────────
   1. Sousou no Frieren 2nd Season                0.276  0.622  0.866  0.3693
   2. Kono Subarashii Sekai ni Shukufuku wo! 3    0.156  0.911  0.577  0.2736
   3. Mushoku Tensei: Isekai Ittara Honki Dasu Part 2  0.141  0.688  0.750  0.2567
   4. Kono Subarashii Sekai ni Shukufuku wo! 2    0.108  0.909  0.577  0.2348
   5. Mushoku Tensei II: Isekai Ittara Honki Dasu  0.090  0.854  0.750  0.2328
   6. Kanon (2006)                                0.152  0.805  0.289  0.2311
   7. Log Horizon                                 0.100  0.914  0.577  0.2294
   8. Mononoke Hime                               0.096  0.727  0.750  0.2245
   9. Dragon Ball Super: Broly                    0.114  0.660  0.577  0.2153
  10. Kono Subarashii Sekai ni Shukufuku wo! Movie

In [14]:
# ── 6. Filtro por gênero ──────────────────────────────────────────
# Apenas animes que tenham ao menos um dos gêneros listados
recomendar(
    'Bocchi the Rock!',
    df_base,
    MATRIZES,
    peso_texto   = 0.80,
    peso_imagem  = 0.10,
    peso_genero  = 0.10,
    metrica      = 'Cosseno',
    n            = 10,
    filtro_genero= ['Music', 'Comedy', 'Slice of Life'],
)


  [Cosseno]  Top 10 para 'Bocchi the Rock!'
  Pesos → Texto: 80% | Imagem: 10% | Gênero: 10%
  Filtro gênero: ['Music', 'Comedy', 'Slice of Life']  (269 candidatos)
   #  Título                                        Txt    Img    Gen   Final
  ──────────────────────────────────────────────────────────────────
   1. K-On! Movie                                 0.202  0.583  0.000  0.2200
   2. Love Live! The School Idol Movie            0.143  0.685  0.000  0.1828
   3. Boku no Kokoro no Yabai Yatsu 2nd Season    0.147  0.615  0.000  0.1790
   4. City Hunter                                 0.129  0.735  0.000  0.1768
   5. Watashi ga Koibito ni Nareru Wake Nai jan, Muri Muri! (※Muri ja Nakatta!?) (TV Special)  0.139  0.655  0.000  0.1767
   6. Lovely★Complex                              0.155  0.523  0.000  0.1763
   7. Hachimitsu to Clover                        0.159  0.464  0.000  0.1732
   8. Shirobako                                   0.131  0.612  0.000  0.1659
   9. Kami nomi zo

{'Cosseno':                                                Title  \
 0                                        K-On! Movie   
 1                   Love Live! The School Idol Movie   
 2           Boku no Kokoro no Yabai Yatsu 2nd Season   
 3                                        City Hunter   
 4  Watashi ga Koibito ni Nareru Wake Nai jan, Mur...   
 5                                     Lovely★Complex   
 6                               Hachimitsu to Clover   
 7                                          Shirobako   
 8                        Kami nomi zo Shiru Sekai II   
 9                             Yojouhan Shinwa Taikei   
 
                                               Genres  sim_texto  sim_imagem  \
 0                              Award Winning, Comedy     0.2022      0.5827   
 1                       Award Winning, Slice of Life     0.1430      0.6847   
 2                                    Comedy, Romance     0.1469      0.6145   
 3                            Action, Co

In [15]:
# ── 7. Métrica única ──────────────────────────────────────────────
recomendar(
    'Fullmetal Alchemist: Brotherhood',
    df_base,
    MATRIZES,
    peso_texto  = 0.80,
    peso_imagem = 0.10,
    peso_genero = 0.10,
    metrica     = 'Jaccard',
    n           = 10,
)


  [Jaccard]  Top 10 para 'Fullmetal Alchemist: Brotherhood'
  Pesos → Texto: 80% | Imagem: 10% | Gênero: 10%
   #  Título                                        Txt    Img    Gen   Final
  ──────────────────────────────────────────────────────────────────
   1. Fullmetal Alchemist                         0.135  1.000  0.800  0.2877
   2. Omae Umasou da na                           0.095  1.000  0.750  0.2508
   3. Tian Guan Cifu                              0.060  1.000  1.000  0.2479
   4. D.Gray-man                                  0.091  1.000  0.750  0.2477
   5. Houseki no Kuni                             0.109  1.000  0.600  0.2470
   6. Sennen Joyuu                                0.100  1.000  0.667  0.2467
   7. Tate no Yuusha no Nariagari                 0.056  1.000  1.000  0.2444
   8. Berserk: Ougon Jidai-hen - Memorial Edition  0.079  1.000  0.800  0.2434
   9. One Piece Film: Strong World                0.085  1.000  0.750  0.2431
  10. One Piece: Gyojin Tou-hen         

{'Jaccard':                                          Title  \
 0                          Fullmetal Alchemist   
 1                            Omae Umasou da na   
 2                               Tian Guan Cifu   
 3                                   D.Gray-man   
 4                              Houseki no Kuni   
 5                                 Sennen Joyuu   
 6                  Tate no Yuusha no Nariagari   
 7  Berserk: Ougon Jidai-hen - Memorial Edition   
 8                 One Piece Film: Strong World   
 9                    One Piece: Gyojin Tou-hen   
 
                                               Genres  sim_texto  sim_imagem  \
 0   Action, Adventure, Award Winning, Drama, Fantasy     0.1346         1.0   
 1                         Action, Adventure, Fantasy     0.0947         1.0   
 2                  Action, Adventure, Drama, Fantasy     0.0598         1.0   
 3                         Action, Adventure, Fantasy     0.0909         1.0   
 4                    Acti

In [16]:
# ── 8. Comparação de pesos ────────────────────────────────────────
# Mostra como o balanço texto/imagem/gênero altera os resultados

ANIME_TESTE = 'The First Slam Dunk'
N           = 5

configs = [
    (1.00, 0.00, 0.00, 'Só texto'),
    (0.80, 0.10, 0.10, 'Texto 80 / Img 10 / Gen 10  ← padrão'),
    (0.60, 0.10, 0.30, 'Texto 60 / Img 10 / Gen 30'),
    (0.50, 0.20, 0.30, 'Texto 50 / Img 20 / Gen 30'),
    (0.00, 0.00, 1.00, 'Só gênero'),
    (0.00, 1.00, 0.00, 'Só imagem'),
]

print(f"Comparando configurações para: '{ANIME_TESTE}'  (top {N})\n")
for wT, wI, wG, label in configs:
    res = recomendar(
        ANIME_TESTE, df_base, MATRIZES,
        peso_texto=wT, peso_imagem=wI, peso_genero=wG,
        metrica='Cosseno', n=N,
    )
    titulos = res['Cosseno']['Title'].tolist() if res else []
    print(f"[{label:<42}]  {titulos}")

Comparando configurações para: 'The First Slam Dunk'  (top 5)


  [Cosseno]  Top 5 para 'The First Slam Dunk'
  Pesos → Texto: 100% | Imagem: 0% | Gênero: 0%
   #  Título                                        Txt    Img    Gen   Final
  ──────────────────────────────────────────────────────────────────
   1. Kuroko no Basket                            0.212  0.837  0.000  0.2124
   2. Ookiku Furikabutte: Natsu no Taikai-hen     0.196  0.701  0.000  0.1958
   3. Slam Dunk                                   0.195  0.867  0.000  0.1949
   4. Kuroko no Basket 3rd Season                 0.192  0.841  0.000  0.1918
   5. Chihayafuru 2                               0.189  0.755  0.408  0.1892
[Só texto                                  ]  ['Kuroko no Basket', 'Ookiku Furikabutte: Natsu no Taikai-hen', 'Slam Dunk', 'Kuroko no Basket 3rd Season', 'Chihayafuru 2']

  [Cosseno]  Top 5 para 'The First Slam Dunk'
  Pesos → Texto: 80% | Imagem: 10% | Gênero: 10%
   #  Título                          

In [17]:
# ── 9. Busca por fragmento ────────────────────────────────────────
# Útil quando você não lembra o nome exato

def buscar_titulo(fragmento, df, n=10):
    """Busca títulos que contenham o fragmento (case-insensitive)."""
    mask = df['Title'].str.contains(fragmento, case=False, na=False)
    found = df['Title'][mask].tolist()
    if not found:
        print(f"Nenhum título encontrado para '{fragmento}'")
    else:
        print(f"{len(found)} resultado(s):")
        for t in found[:n]: print(f"  • {t}")
    return found

buscar_titulo('jojo', df_base)
print()
buscar_titulo('attack on', df_base)

9 resultado(s):
  • Steel Ball Run: JoJo no Kimyou na Bouken
  • JoJo no Kimyou na Bouken Part 5: Ougon no Kaze
  • JoJo no Kimyou na Bouken Part 4: Diamond wa Kudakenai
  • JoJo no Kimyou na Bouken Part 6: Stone Ocean Part 3
  • JoJo no Kimyou na Bouken Part 3: Stardust Crusaders - Egypt-hen
  • JoJo no Kimyou na Bouken Part 3: Stardust Crusaders
  • JoJo no Kimyou na Bouken Part 6: Stone Ocean
  • JoJo no Kimyou na Bouken Part 6: Stone Ocean Part 2
  • JoJo no Kimyou na Bouken (TV)

Nenhum título encontrado para 'attack on'


[]

In [18]:
# ── 10. Use aqui ─────────────────────────────────────────────────

recomendar(
    titulo       = 'Steins;Gate',
    df           = df_base,
    matrizes     = MATRIZES,
    peso_texto   = 0.80,
    peso_imagem  = 0.10,
    peso_genero  = 0.10,
    metrica      = None,    # None = todas as métricas
    n            = 10,
    filtro_genero= None,    # ex: ['Sci-Fi', 'Suspense'] para filtrar candidatos
)


  [Cosseno]  Top 10 para 'Steins;Gate'
  Pesos → Texto: 80% | Imagem: 10% | Gênero: 10%
   #  Título                                        Txt    Img    Gen   Final
  ──────────────────────────────────────────────────────────────────
   1. Steins;Gate 0                               0.637  0.730  1.000  0.6825
   2. Steins;Gate: Kyoukaimenjou no Missing Link - Divide By Zero  0.515  0.824  0.817  0.5759
   3. Steins;Gate Movie: Fuka Ryouiki no Déjà vu  0.446  0.842  0.817  0.5222
   4. Steins;Gate: Oukoubakko no Poriomania       0.450  0.747  0.408  0.4755
   5. Kaoru Hana wa Rin to Saku                   0.115  0.786  0.408  0.2115
   6. Re:Zero kara Hajimeru Isekai Seikatsu 2nd Season  0.070  0.845  0.667  0.2073
   7. Evangelion Movie 2: Ha                      0.044  0.704  1.000  0.2055
   8. Gnosia                                      0.036  0.943  0.817  0.2052
   9. Shiguang Dailiren                           0.063  0.867  0.667  0.2037
  10. Shin Evangelion Movie:||         

{'Cosseno':                                                Title  \
 0                                      Steins;Gate 0   
 1  Steins;Gate: Kyoukaimenjou no Missing Link - D...   
 2         Steins;Gate Movie: Fuka Ryouiki no Déjà vu   
 3              Steins;Gate: Oukoubakko no Poriomania   
 4                          Kaoru Hana wa Rin to Saku   
 5   Re:Zero kara Hajimeru Isekai Seikatsu 2nd Season   
 6                             Evangelion Movie 2: Ha   
 7                                             Gnosia   
 8                                  Shiguang Dailiren   
 9                           Shin Evangelion Movie:||   
 
                                    Genres  sim_texto  sim_imagem  sim_genero  \
 0                 Drama, Sci-Fi, Suspense     0.6369      0.7297      1.0000   
 1                        Sci-Fi, Suspense     0.5148      0.8243      0.8165   
 2                           Drama, Sci-Fi     0.4455      0.8418      0.8165   
 3                          Comedy, 

## Notas técnicas

### Por que normalizar L2 antes do cosseno?
Garante que todos os vetores tenham magnitude 1, tornando o produto escalar equivalente ao cosseno. Sem isso, documentos mais longos (synopsis maior) teriam vetores com magnitude maior e dominariam o score.

### Por que as matrizes são pré-calculadas?
Calcular `cosine_similarity` para 1050×1050 leva ~1s. Pré-calcular e reutilizar evita recomputar a cada consulta.

### Gêneros como one-hot normalizado
`str.get_dummies(sep=', ')` cria uma coluna binária por gênero (19 no total). Normalizar por L2 garante que o peso da modalidade gênero não seja distorcido pelo número de gêneros de cada anime.

### Escolha de pesos recomendada
| Objetivo | texto | imagem | gênero |
|---|---|---|---|
| Fidelidade narrativa | 0.85 | 0.05 | 0.10 |
| Equilíbrio (padrão) | 0.80 | 0.10 | 0.10 |
| Énfase em gênero | 0.60 | 0.10 | 0.30 |
| Visual matching | 0.50 | 0.40 | 0.10 |

### Próximo passo
Substituir o TF-IDF por embeddings de linguagem (ex: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`) para capturar semântica mais profunda da synopsis.

In [23]:
import json
import numpy as np

# Função para tratar matrizes (limpar NaNs e reduzir precisão para o HTML não ficar gigante)
def prep_matrix(mat, decimals=4):
    mat_limpa = np.nan_to_num(mat)
    return np.round(mat_limpa, decimals).tolist()

print("Empacotando dados e TODAS as matrizes de métricas...")
metadata = df_base[['Title', 'Genres', 'Image_URL', 'Synopsis', 'Score']].fillna('').to_dict(orient='records')
titulos = df_base['Title'].tolist()

# Criamos uma estrutura para levar todas as métricas calculadas no Python
todas_matrizes = {}
for metrica in ['Cosseno', 'Euclidiana', 'Manhattan', 'Jaccard']:
    if metrica in MATRIZES:
        sim_texto, sim_imagem, sim_genero = MATRIZES[metrica]
        todas_matrizes[metrica] = {
            'texto': prep_matrix(sim_texto),
            'imagem': prep_matrix(sim_imagem),
            'genero': prep_matrix(sim_genero)
        }

data_dict = {
    'titulos': titulos,
    'metadata': metadata,
    'matrizes': todas_matrizes
}

print("Convertendo para JSON (pode levar alguns segundos devido ao volume)...")
data_json = json.dumps(data_dict).replace("</", "<\\/")

# --- TEMPLATE DO ARQUIVO HTML ---
html_template = """<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Recomendador Multimodal de Animes</title>
    <style>
        :root { --bg: #0f172a; --card: #1e293b; --accent: #38bdf8; --text: #f8fafc; }
        body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; background: var(--bg); color: var(--text); margin: 0; padding: 20px; }
        .container { max-width: 1000px; margin: 0 auto; }
        h1 { text-align: center; color: var(--accent); font-weight: 800; margin-bottom: 30px;}
        .controls { background: var(--card); padding: 25px; border-radius: 12px; margin-bottom: 30px; box-shadow: 0 10px 15px -3px rgba(0,0,0,0.5); }
        .form-group { margin-bottom: 20px; }
        label { display: block; margin-bottom: 8px; font-weight: 600; font-size: 14px; color: #cbd5e1; }
        input[type="text"], select { width: 100%; padding: 12px; border-radius: 6px; border: 1px solid #334155; background: #0f172a; color: white; font-size: 15px; box-sizing: border-box; outline: none; }
        input[type="text"]:focus, select:focus { border-color: var(--accent); }
        .sliders { display: flex; gap: 20px; flex-wrap: wrap; }
        .slider-box { flex: 1; min-width: 150px; }
        input[type="range"] { width: 100%; accent-color: var(--accent); }
        button { background: var(--accent); color: #0f172a; border: none; padding: 14px 20px; font-size: 16px; border-radius: 6px; cursor: pointer; width: 100%; font-weight: bold; transition: opacity 0.2s; margin-top: 10px; }
        button:hover { opacity: 0.9; }
        .results { display: grid; grid-template-columns: repeat(auto-fill, minmax(180px, 1fr)); gap: 20px; }
        .card { background: var(--card); border-radius: 10px; overflow: hidden; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.5); transition: transform 0.2s; display: flex; flex-direction: column; }
        .card:hover { transform: translateY(-5px); }
        .card img { width: 100%; height: 260px; object-fit: cover; }
        .card-content { padding: 15px; flex-grow: 1; display: flex; flex-direction: column; }
        .card-title { font-size: 15px; font-weight: bold; margin-bottom: 8px; line-height: 1.3; flex-grow: 1; }
        .card-score { color: #94a3b8; font-size: 13px; margin-bottom: 4px; }
        .sim-bar { background: #334155; border-radius: 999px; height: 6px; margin-top: auto; overflow: hidden; }
        .sim-fill { background: var(--accent); height: 100%; border-radius: 999px; }
        .weight-warning { text-align: center; font-weight: bold; margin-top: 10px; color: #f43f5e; font-size: 14px; }
    </style>
</head>
<body>
    <div class="container">
        <h1>Recomendador Multimodal Interativo</h1>
        <div class="controls">
            <div class="form-group">
                <label for="anime-input">🔍 Escolha um Anime (Base):</label>
                <input type="text" id="anime-input" list="anime-list" placeholder="Comece a digitar o nome do anime...">
                <datalist id="anime-list"></datalist>
            </div>
            
            <div class="form-group">
                <label for="metric-select">📏 Métrica de Similaridade:</label>
                <select id="metric-select">
                    <option value="Cosseno">Cosseno</option>
                    <option value="Euclidiana">Euclidiana</option>
                    <option value="Manhattan">Manhattan</option>
                    <option value="Jaccard">Jaccard</option>
                </select>
            </div>

            <div class="form-group">
                <label for="genre-filter">🎭 Filtro de Gênero (Opcional, separe por vírgula):</label>
                <input type="text" id="genre-filter" placeholder="Ex: Action, Comedy, Drama...">
            </div>

            <div class="sliders form-group">
                <div class="slider-box">
                    <label>📝 Peso Texto: <span id="val-texto">0.80</span></label>
                    <input type="range" id="peso-texto" min="0" max="1" step="0.05" value="0.80">
                </div>
                <div class="slider-box">
                    <label>🖼️ Peso Imagem: <span id="val-imagem">0.10</span></label>
                    <input type="range" id="peso-imagem" min="0" max="1" step="0.05" value="0.10">
                </div>
                <div class="slider-box">
                    <label>🏷️ Peso Gênero: <span id="val-genero">0.10</span></label>
                    <input type="range" id="peso-genero" min="0" max="1" step="0.05" value="0.10">
                </div>
            </div>
            
            <div class="weight-warning" id="soma-status">Soma atual dos pesos: 100%</div>

            <button onclick="gerarRecomendacoes()">Gerar Recomendações</button>
        </div>

        <div class="results" id="results-container"></div>
    </div>

    <script>
        const db = __JSON_DATA__;
        
        // Popula o Autocomplete
        const datalist = document.getElementById('anime-list');
        db.titulos.forEach(titulo => {
            let option = document.createElement('option');
            option.value = titulo;
            datalist.appendChild(option);
        });

        // Atualiza textos dos sliders e calcula a soma em tempo real
        function atualizarSoma() {
            const pText = parseFloat(document.getElementById('peso-texto').value);
            const pImg = parseFloat(document.getElementById('peso-imagem').value);
            const pGen = parseFloat(document.getElementById('peso-genero').value);
            const sumP = pText + pImg + pGen;
            
            const statusDiv = document.getElementById('soma-status');
            const pct = Math.round(sumP * 100);
            
            statusDiv.innerText = `Soma atual dos pesos: ${pct}%`;
            if (Math.abs(sumP - 1.0) > 0.001) {
                statusDiv.style.color = '#f43f5e';
            } else {
                statusDiv.style.color = '#10b981';
            }
        }

        ['texto', 'imagem', 'genero'].forEach(tipo => {
            const input = document.getElementById(`peso-${tipo}`);
            const span = document.getElementById(`val-${tipo}`);
            input.addEventListener('input', () => {
                span.innerText = parseFloat(input.value).toFixed(2);
                atualizarSoma();
            });
        });

        atualizarSoma();

        function gerarRecomendacoes() {
            const animeBusca = document.getElementById('anime-input').value;
            const targetIdx = db.titulos.indexOf(animeBusca);
            
            if (targetIdx === -1) {
                alert("Anime não encontrado! Por favor, selecione um anime da lista (Autocompletar).");
                return;
            }

            const pText = parseFloat(document.getElementById('peso-texto').value);
            const pImg = parseFloat(document.getElementById('peso-imagem').value);
            const pGen = parseFloat(document.getElementById('peso-genero').value);
            const sumP = pText + pImg + pGen;

            // Trava de 100% dos pesos
            if (Math.abs(sumP - 1.0) > 0.001) {
                alert(`Aviso: A soma dos pesos deve dar exatamente 100% para rodar! Atual: ${Math.round(sumP * 100)}%`);
                return;
            }

            // Filtro de gênero
            const filterStr = document.getElementById('genre-filter').value.trim();
            let generosFiltro = filterStr ? filterStr.split(',').map(g => g.trim().toLowerCase()).filter(g => g) : [];

            // Captura a métrica escolhida no dropdown
            const metricaSelecionada = document.getElementById('metric-select').value;

            let resultados = [];

            for(let i=0; i<db.titulos.length; i++) {
                if(i === targetIdx) continue;

                if(generosFiltro.length > 0) {
                    const genresAnime = (db.metadata[i].Genres || "").toLowerCase();
                    const hasGenre = generosFiltro.some(g => genresAnime.includes(g));
                    if(!hasGenre) continue;
                }

                // BUSCA DIRETO DA MATRIZ DO PYTHON CORRESPONDENTE À MÉTRICA SELECIONADA
                const sText = db.matrizes[metricaSelecionada]['texto'][targetIdx][i];
                const sImg = db.matrizes[metricaSelecionada]['imagem'][targetIdx][i];
                const sGen = db.matrizes[metricaSelecionada]['genero'][targetIdx][i];

                const score = (pText*sText + pImg*sImg + pGen*sGen);
                resultados.push({ idx: i, score: score });
            }

            resultados.sort((a, b) => b.score - a.score);
            renderizar(resultados.slice(0, 10));
        }

        function renderizar(tops) {
            const container = document.getElementById('results-container');
            container.innerHTML = '';
            
            if(tops.length === 0) {
                container.innerHTML = '<p style="grid-column: 1/-1; text-align: center; color: #94a3b8;">Nenhum anime encontrado com esses critérios.</p>';
                return;
            }

            tops.forEach(item => {
                const meta = db.metadata[item.idx];
                const pct = (item.score * 100).toFixed(1);
                
                const card = document.createElement('div');
                card.className = 'card';
                card.innerHTML = `
                    <img src="${meta.Image_URL || 'https://via.placeholder.com/200x280?text=Sem+Imagem'}" alt="${meta.Title}">
                    <div class="card-content">
                        <div class="card-title">${meta.Title}</div>
                        <div class="card-score">Score MAL: ⭐ ${meta.Score}</div>
                        <div class="card-score">Similaridade: ${pct}%</div>
                        <div class="sim-bar"><div class="sim-fill" style="width: ${pct}%"></div></div>
                    </div>
                `;
                container.appendChild(card);
            });
        }
    </script>
</body>
</html>"""

# Injetar os dados e salvar
html_final = html_template.replace('__JSON_DATA__', data_json)

with open('app_recomendador.html', 'w', encoding='utf-8') as f:
    f.write(html_final)

print("✅ Arquivo 'app_recomendador.html' gerado com sucesso com TODAS as métricas corrigidas!")

Empacotando dados e TODAS as matrizes de métricas...
Convertendo para JSON (pode levar alguns segundos devido ao volume)...
✅ Arquivo 'app_recomendador.html' gerado com sucesso com TODAS as métricas corrigidas!
